# 01 — Business & Data Understanding
**Tujuan:** audit dataset sanitasi, memahami distribusi kondisi, dan mengeksplorasi pola inventaris.

Dataset yang disertakan tidak memuat `plat_nomor`. Seluruh hasil adalah analisis snapshot periode 2025, bukan kesiapan operasional real-time.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "processed" / "kendaraan_2025_sanitized.csv"
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Dimensi:", df.shape)
print("\nTipe data:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing value:")
display(df.isna().sum().rename("missing").to_frame())
print("\nDuplikasi identik:", df.duplicated().sum())
print("\nPeriode:")
display(df["periode_data"].value_counts(dropna=False))
print("\nDistribusi target:")
display(pd.concat([
    df["Kondisi"].value_counts(dropna=False).rename("jumlah"),
    (df["Kondisi"].value_counts(normalize=True, dropna=False)*100).round(2).rename("persen")
], axis=1))

## Pemeriksaan kualitas dan kategori

Dataset bersifat dominan kategorikal. Karena itu, deteksi outlier numerik konvensional seperti IQR tidak menjadi fokus. Audit dilakukan terhadap nilai kosong, duplikasi, kategori, dan konsistensi hierarki administratif.

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].nunique(dropna=True)} kategori unik")
print("\nNilai kategori target:", sorted(df["Kondisi"].dropna().unique().tolist()))
print("\nKonsistensi label Kondisi:")
display(df["Kondisi"].value_counts(dropna=False))

In [ ]:
target_counts = df["Kondisi"].value_counts()
ax = target_counts.reindex(["BAIK", "RUSAK"]).plot(kind="bar", rot=0)
ax.set_title("Distribusi Kondisi Kendaraan")
ax.set_xlabel("Kondisi tercatat")
ax.set_ylabel("Jumlah entri")
plt.tight_layout()
plt.show()
print("Rasio BAIK terhadap RUSAK:", round(target_counts.get("BAIK", 0)/target_counts.get("RUSAK", 1), 2))

In [ ]:
wilayah = pd.crosstab(df["wilayah"], df["Kondisi"])
wilayah["TOTAL"] = wilayah.sum(axis=1)
wilayah["RUSAK_PCT"] = (wilayah.get("RUSAK", 0) / wilayah["TOTAL"] * 100).round(2)
display(wilayah.sort_values("RUSAK_PCT", ascending=False))
plot_data = wilayah.sort_values("RUSAK_PCT", ascending=False)
ax = plot_data["RUSAK_PCT"].plot(kind="bar", rot=70)
ax.set_title("Persentase RUSAK menurut Wilayah")
ax.set_ylabel("RUSAK (%)")
plt.tight_layout()
plt.show()

In [ ]:
jenis = pd.crosstab(df["jenis_kendaraan"], df["Kondisi"])
jenis["TOTAL"] = jenis.sum(axis=1)
jenis["RUSAK_PCT"] = (jenis.get("RUSAK", 0) / jenis["TOTAL"] * 100).round(2)
display(jenis.sort_values(["RUSAK_PCT", "TOTAL"], ascending=[False, False]).head(15))
display(df["jenis_kendaraan"].value_counts().head(10).rename("jumlah").to_frame())

## Asosiasi kategorikal

Cramér’s V mengukur kekuatan asosiasi antara dua variabel kategorikal, bukan sebab-akibat. Banyaknya kategori dan sel dengan frekuensi harapan kecil dapat memengaruhi interpretasi. Nilai p dari chi-square hanya ditampilkan sebagai eksplorasi.

In [ ]:
def cramers_v(table):
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    r, k = table.shape
    if n == 0 or min(r, k) <= 1:
        return np.nan
    return np.sqrt(chi2 / (n * min(k-1, r-1)))

rows = []
for col in ["wilayah", "kecamatan", "kelurahan", "jenis_kendaraan", "klasifikasi", "fungsi", "penempatan"]:
    tab = pd.crosstab(df[col], df["Kondisi"])
    chi2, p, dof, expected = chi2_contingency(tab, correction=False)
    rows.append({"variabel": col, "kategori": df[col].nunique(),
                 "cramers_v": cramers_v(tab), "chi2_p_eksploratif": p,
                 "sel_expected_lt5_pct": (expected < 5).mean()*100})
assoc = pd.DataFrame(rows).sort_values("cramers_v", ascending=False)
display(assoc.round(4))

## Interpretasi awal

- Kelas BAIK dominan; akurasi saja tidak cukup untuk menilai model.
- Proporsi RUSAK perlu dibaca bersama jumlah entri pada setiap kelompok.
- `plat_nomor` tidak tersedia dalam data portofolio sanitasi dan tidak boleh digunakan sebagai prediktor default.
- Tanpa koordinat, analisis per wilayah administratif tidak dapat disebut klasterisasi spasial berbasis jarak.
- Hasil asosiasi tidak membuktikan bahwa lokasi atau jenis kendaraan menyebabkan kerusakan.